In [ ]:
import matplotlib.pyplot as plt, requests, os, pandas as pd

In [ ]:
# need to setup the code inside data_ingestion.py
from dotenv import load_dotenv
load_dotenv() #this gets the API key from .env file so we use it.
API_KEY = os.getenv("API_KEY")


In [ ]:
events_url = f'https://app.ticketmaster.com/discovery/v2/events.json?apikey={API_KEY}'
# classifications_url = (
#     'https://app.ticketmaster.com/discovery/v2/'
#     f"classifications.json?source=&apikey={API_KEY}&locale=en-us"
#       )

# genre_url = f'https://app.ticketmaster.com/discovery/v2/classifications/genres'
# attractions_url = f'https://app.ticketmaster.com/discovery/v2/attractions.json?apikey={API_KEY}'

In [ ]:
# attractions_params_action_api = {
#     'preferredCountry': ['us'], 
#     'source': ["ticketmaster"],
#     'classificationName': 'music'
# }
events_params_action_api = {
    'startDateTime': '2025-05-01T18:00:00-05:00',
    'endDateTime': '2027-12-31T17:00:00-05:00',
    'classificationName': 'music',
    # 'dmaId': '345', 
    'city': 'New York',
    'size': '20', 
    'page': '5'
}

In [ ]:
# r = requests.get(attractions_url, params=attractions_params_action_api)

r = requests.get(events_url, params=events_params_action_api)
print(r.status_code)

# r = requests.get(classifications_url)
# classifications_data = r.json()
# print(classifications_data)

In [ ]:
data=r.json()

In [ ]:
# with open('output.txt', 'wt') as f:
#     print(data, file=f)

In [ ]:
print(data.keys())

In [ ]:
print(data['page'])

In [ ]:
print(len(data['_embedded']['events']))

In [ ]:
print(data)

In [ ]:
# print(data['_embedded'].keys())
# print(type(data['_embedded']['events'])) #list
# print(data['_embedded']['events'])
# event_name = data['_embedded']['events'][1]['name']

# event_type = data['_embedded']['events'][0]['type']


# # event_id = data['_embedded']['events'][0]['id']
# # print(event_id)
# # genre_id = data['_embedded']['events'][0]['classifications'][0]['genre']['id']
# # genre_type = data['_embedded']['events'][0]['classifications'][0]['genre']['name']

# print(event_name)
# print(event_type)

In [ ]:
# some_segment_id = data['_embedded']['classifications'][0]['segment']['id']
# some_id = data['_embedded']['classifications'][0]['segment']['id']

In [ ]:
# print(data['_embedded']['events'])

In [ ]:
# genre_id = data['_embedded']['events'][0]['classifications'][0]['genre']['id']
# genre_type = data['_embedded']['events'][0]['classifications'][0]['genre']['name']


In [ ]:
# print(genres)

In [ ]:
# print(classifications_data['_embedded']['classifications']) #list


In [ ]:
# classifications_data['_embedded']['classifications'][0]['segment']['name']

In [ ]:
new_york_url = f'https://app.ticketmaster.com/discovery/v2/events.json?apikey={API_KEY}&=&city=New%20York&stateCode=NY&countryCode=US'


ny_data = requests.get(new_york_url)
ny_data.json()
with open('output.txt', 'wt') as f:
    print(ny_data.json(), file=f)

In [ ]:
data['_embedded']['events']


### Creating initial dataframe for data exploration ### 

In [ ]:
df = pd.DataFrame(data['_embedded']['events'])
df

In [ ]:
df.info()

In [ ]:
accessibility = df['accessibility']



In [ ]:
accessibility.iloc[13]


### For-loop to grab the value of Accessibility in nested dictionary encased in a list ### 

In [ ]:
for item in accessibility:
    for location, value in item.items():
        print(f'Ticket Limit: ',value)


### Creating variable to refer to 'products' column ### 

In [ ]:
product_name = df['products']


### Creating for loop to grab value of PRODUCTS column ### 

In [ ]:
for item in product_name:
    for n in item:
        print(n['name'])

### Re-assigning the output of above for-loop to PRODUCTS column to reflect new changes. ### 

In [ ]:
df['products'] = n['name']


### List comprehension using for loop and re-assigning value to Accessibility column ###

In [ ]:
df['accessibility'] = [f'Ticket Limit: {value}'
                       for item in accessibility
                        for location, value in item.items()]

# list comprehension to loop the 'accessibility' column to get Ticket Limit dictionary.

In [ ]:
df
df.iloc[0,9]

In [ ]:
df.iloc[1,4]

In [ ]:
df.rename(columns={
    '_embedded': 'Venue Name', 
    'name': 'Name',
    'type': 'Type', 
    'id': 'ID',
    'products': 'PRODUCTS', 
    'accessibility': 'Accessibility',
    'classifications': 'Genre'

    }, inplace=True)

In [ ]:
df

#### Creating variable to 'Genre' column ####

In [ ]:
genre = df['Genre']
genre

#### For loop to pull Genre name out of dictionary list ####

In [ ]:
for x in genre: #x is the dictionary
    for item in x:
        print(item['genre']['name']) #loop through Genre list to pull Genre name.


#### list comprehension using for loop and re-assigning the value to 'Genre' column ####

In [ ]:
df['Genre'] = [item['genre']['name'] 
                for x in genre
                for item in x]


#### Creating variable for 'Venue Name' #### 

In [ ]:
venue_name = df['Venue Name']


#### For loop to grab the venue name from dictionary in a list #### 

In [ ]:
for x in venue_name: # to loop through Venue Name
    for y in x['venues']:
        print(y['name'])

#### List comprehension using for loop and reassigning the value back to 'Venue Name' column. ####

In [ ]:
df['Venue Name'] = [y['name'] 
                    for x in venue_name
                    for y in x['venues']]

In [ ]:
df.info()

### Here, we're implementing a function `getAllPages()` for pagination, in order to retrieve additional pages from the server. We're pulling 1000 rows of data and returning a dataframe. ### 

In [ ]:
def getAllPages(pages=10, per_page=100):
    # url = f'https://app.ticketmaster.com/discovery/v2/events.json?apikey={API_KEY}'
    
    events_params_action_api = {
    'startDateTime': '2025-05-01T18:00:00-05:00',
    'endDateTime': '2027-12-31T17:00:00-05:00',
    'classificationName': 'music',
    # 'dmaId': '345', 
    'city': 'New York',
    'size': '200', 
    'page': '100'
    }
    
    all_records = []

    # url = f'https://app.ticketmaster.com/discovery/v2/events.json?countryCode=US&apikey={API_KEY}'
    url = f'https://app.ticketmaster.com/discovery/v2/events.json?apikey={API_KEY}'

    for page in range(1, pages + 1):         
        response = requests.get(
            url,
            params={**events_params_action_api, 'page': page, 'size': per_page}
        )
        response.raise_for_status()
        records = response.json()
        #(data['_embedded']['events'])
        all_records.extend(records['_embedded']['events'])
        
    return pd.DataFrame(all_records) 

### Created a variable `more_events` to call the pagination function `getAllPages()` ### 

In [ ]:
more_events= getAllPages()

In [ ]:
more_events.rename(columns= {
    'name':'NAME', 
    'type': 'TYPE', 
    'id': 'ID',
    'classifications': 'GENRE', 
    'accessibility': 'ACCESSIBILITY', 
    'ticketLimit': 'TICKET_LIMIT', 
    '_embedded': 'VENUE_NAME'
}
)

In [ ]:
more_events.info()

In [ ]:
more_events

### Created `data_ex` variable to filter for columns that we don't need and renaming columns

In [ ]:
data_ex = more_events.loc[:, ~more_events.columns.isin(['images','doorsTimes', '_links', 'linkMoreInfo', 'outlets', 'images', 'test','pleaseNote', 'seatmap','nameOrigin', 'promoters', 'url', 'priceRanges'])]
# excluding several columns from dataframe

In [ ]:
data_ex
# data_ex.dtypes
# print(type(data_ex))


In [ ]:
data_ex.rename(columns= {
    'name':'NAME', 
    'type': 'TYPE', 
    'id': 'ID',
    'classifications': 'GENRE', 
    'promoter': 'PROMOTER',
    'accessibility': 'ACCESSIBILITY', 
    'ticketLimit': 'TICKET_LIMIT', 
    '_embedded': 'VENUE_NAME',
    'products': 'PRODUCTS',
    'ticketLimit': 'TICKET_LIMIT',
    'info': 'INFO'


}, inplace=True
)

In [ ]:
product_name = data_ex['PRODUCTS']


In [ ]:
print(data_ex['PRODUCTS'].isna().sum())

### This code which worked for the initial 20 rows of data, won't work for 1000 rows of data, since there's some missing products which results in NaN values. ###

In [ ]:
# product_name = data_ex['products']
# for item in product_name:
#     for n in item:
#         print(n['name'])


In [ ]:
product_name = n['name']

In [ ]:
data_ex 

#### Dropping NAN values for Products, Accessibility, Genre column, Venue Name column and making a copy while retaining original column values. #### 

In [ ]:
data_clean = data_ex.dropna(subset = ['PRODUCTS', 'ACCESSIBILITY', 'GENRE', 'VENUE_NAME']).copy()

In [ ]:
data_clean.rename(columns= {
    'name':'NAME', 
    'type': 'TYPE', 
    'id': 'ID',
    'classifications': 'GENRE', 
    'promoter': 'PROMOTER',
    'ticketLimit': 'TICKET_LIMIT', 
    '_embedded': 'VENUE_NAME',
    'ticketLimit': 'TICKET_LIMIT',
    'info': 'INFO',
    'products': 'PRODUCTS',
    'accessibility': 'ACCESSIBILITY'


}, inplace=True
)

#### List comprehension to create new dataframe after dropped NaN values to create new column for 'Products_Filtered' ####

In [ ]:
data_clean['Products_Filtered'] = [
    [n['name'] for n in item] for item in data_clean['PRODUCTS']
] #list of products name

data_clean['Products_Filtered'] = [
    ', '.join([n['name'] for n in item]) for item in data_clean['PRODUCTS']
]
#converted list of products using comma separator

In [ ]:
data_clean['VENUE_NAME_Filtered'] = [
    name['venues'][0]['name']
    for name in data_clean['VENUE_NAME']
    
    
]
data_clean['Accessibility_Filtered'] = [
    f"Ticket Limit: {item.get('ticketLimit', 'N/A')}" for item in data_clean['ACCESSIBILITY']
]


data_clean['Genre_Filtered'] = [item['genre']['name']
                                for x in data_clean['GENRE']
                                for item in x
    ]

In [ ]:
data_clean

In [ ]:
data_clean.info()

##### *Used AI-generated code for filtering* #### 

In [ ]:
data_clean['ACCESSIBILITY'].apply(lambda x: list(x.keys())).value_counts()

In [ ]:
# data_clean.rename(columns= {
#     'name':'NAME', 
#     'type': 'TYPE', 
#     'id': 'ID',
#     'classifications': 'GENRE', 
#     'promoter': 'PROMOTER',
#     'ticketLimit': 'TICKET_LIMIT', 
#     '_embedded': 'VENUE_NAME',
#     'ticketLimit': 'TICKET_LIMIT',
#     'info': 'INFO',
#     'products': 'PRODUCTS',
#     'accessibility': 'ACCESSIBILITY'


# }, inplace=True
# )

#### Filter for Genre ####

In [ ]:
data_cleaned = more_events.loc[:, ~more_events.columns.isin(['images','doorsTimes', '_links', 'linkMoreInfo', 'outlets', 'images', 'test','pleaseNote', 'seatmap','nameOrigin', 'promoters', 'url', 'priceRanges'])]

### Code for looping inside Accessibility column and removing NaN values using list comprehension in data_clean dataframe ###

In [ ]:
# data_clean['Accessibility_Filtered'] = [
#     f'Ticket Limit: {value}'
#     for item in accessibility
#     for location, value in item.items()]

print(data_ex['accessibility'].isna().sum())


### Code for looping inside Genre column and removing NaN values 
### using list comprehension in data_clean dataframe ###

In [ ]:
for x in genre: #x is the dictionary
    for item in x:
        print(item['genre']['name']) 
        #loop through Genre list to pull Genre name.

In [ ]:
name = data_ex['products'].iloc[12]

In [ ]:
data_ex['_embedded']
venue_name = data_ex['_embedded']
venue_name

#### For loop to retrieve Venue name from dictionary ####

In [ ]:
data_ex = more_events.loc[:, ~more_events.columns.isin(['images','doorsTimes', '_links', 'linkMoreInfo', 'outlets', 'images', 'test','pleaseNote', 'seatmap','nameOrigin', 'promoters', 'url', 'priceRanges', 'products', 'accessibility'])]
# excluding several columns from dataframe
data_ex